In [2]:
import os

import matplotlib.pyplot as plt
import pandas as pd

%run analysis_utils.py
%run fig2_plot_utils.py

RESULTS_PATH = os.path.expanduser(
    "~/scFM_eval/results/embedding_bootstrap/embedding.metrics.bootstrap.csv"
)
GROUP_ORDER = ["Baseline", "Geneformer", "Other", "scGPT"]
PLOT_DIR = "./plots"

results_10_runs = pd.read_csv(RESULTS_PATH)
results_10_runs["method"] = results_10_runs["model"]

exclude = (
    results_10_runs.model_display.str.contains("continue")
    | results_10_runs.model_display.str.contains("Full")
    | results_10_runs.model_display.isin(["PCA [100]", "PCA [50]"])
)
results_10_runs = results_10_runs.loc[~exclude].copy()


In [3]:
METRIC_DISPLAY_NAMES = {
    "NMI_cluster/label": "Cluster–Label NMI",
    "ARI_cluster/label": "Cluster–Label ARI",
    "ASW_label": "Cell-Type Separation",
    "graph_conn": "Graph Connectivity",
    "ASW_batch": "Batch Mixing ASW",
    "ASW_label/batch": "Biology–Batch ASW Ratio",
    "batch_ASW": "Batch Mixing Score",
    "PCR_batch": "Batch Effect PCR",
    "iLISI": "Batch Diversity",
    "cLISI": "Cell-Type Purity",
    "kBET": "Batch Mixing",
    "avg_bio": "Overall Biology Score",
    "knn_label_purity": "Neighbor Label Purity",
    "knn_batch_purity": "Neighbor Batch Purity",
    "batch_pred_acc": "Batch Predictability",
    "kNN_label_acc": "kNN Label Accuracy",
    "kNN_label_f1_macro": "kNN Label F1",
}

results_10_runs_renamed = results_10_runs.rename(columns=METRIC_DISPLAY_NAMES)
metrics = list(METRIC_DISPLAY_NAMES.values())


In [4]:
results_10_runs_renamed.model_display.value_counts()

model_display
HVG               10
STATE             10
scFoundation      10
GF-V2 [cancer]    10
GF-V2-Deep        10
GF-V1             10
GF-V2             10
scGPT [cancer]    10
scGPT             10
CellPLM           10
SCimiarity        10
PCA [20]          10
scVI              10
scConcept         10
Nicheformer       10
Name: count, dtype: int64

In [116]:
def prep_plot_df(df, source_col, plot_col):
    out = df[["model_display", source_col, "group"]].copy()
    out.columns = ["model", plot_col, "group"]
    return out


def plot_metric(df, metric_col, ylim=None, save_path=None):
    kwargs = {
        "group_order": GROUP_ORDER,
        "point_size": 2.8,
        "point_alpha": 0.55,
    }
    if ylim is not None:
        kwargs["ylim"] = ylim
    if save_path is not None:
        kwargs["save_path"] = save_path
    return plot_metric_grouped_single_axis(df, metric_col=metric_col, **kwargs)


In [117]:
os.makedirs(PLOT_DIR, exist_ok=True)

# HIGHLIGHT_PLOTS = [
#     ("NMI_cluster/label", "NMI", (0.4, 1.0), "nmi3.png"),
#     ("ARI_cluster/label", "ARI", (0.2, 1.0), "ari3.png"),
#     ("ASW_label", "ASW", (0.3, 0.7), "asw3.png"),
#     ("kNN_label_f1_macro", "F1", (0.6, 1.0), "kNN_label_f1_macro.png"),
# ]

# for source_col, plot_col, ylim, filename in HIGHLIGHT_PLOTS:
#     metric_df = prep_plot_df(results_10_runs, source_col, plot_col)
#     fig, ax = plot_metric(
#         metric_df,
#         metric_col=plot_col,
#         ylim=ylim,
#         save_path=os.path.join(PLOT_DIR, filename),
#     )
#     plt.close(fig)


In [118]:
for metric_name in metrics:
    metric_df = prep_plot_df(results_10_runs_renamed, metric_name, metric_name)
    fig, ax = plot_metric(
        metric_df,
        metric_col=metric_name,
        save_path=os.path.join(PLOT_DIR, f"{metric_name}.png"),
    )
    plt.close(fig)
